<a href="https://colab.research.google.com/github/UmymaM/ml-dl-cv-fundamentals/blob/main/emotion-detection/train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [117]:
import os, cv2, torch, torchvision
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import datasets,transforms
import torch.nn as nn
import zipfile as zf

In [118]:
!kaggle datasets download -d ananthu017/emotion-detection-fer

Dataset URL: https://www.kaggle.com/datasets/ananthu017/emotion-detection-fer
License(s): CC0-1.0
emotion-detection-fer.zip: Skipping, found more recently modified local copy (use --force to force download)


In [119]:
zipfile=zf.ZipFile('emotion-detection-fer.zip') #unzipping the file
zipfile.extractall()
zipfile.close()

In [120]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


EDA

In [121]:
path="/content/train"
for file in os.listdir(path):
    folder_path=os.path.join(path,file)
    print(f"{file} :{len(os.listdir(folder_path))}")
    images=os.listdir(folder_path)
    image_path=os.path.join(folder_path,images[0])
    img=Image.open(image_path)
    img.show()
    print(img.size)

happy :7215
(48, 48)
surprised :3171
(48, 48)
sad :4830
(48, 48)
neutral :4965
(48, 48)
angry :3995
(48, 48)
disgusted :436
(48, 48)
fearful :4097
(48, 48)


In [122]:
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, TensorDataset

In [123]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(3),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [124]:
dataset=datasets.ImageFolder(root=path, transform=train_transform)

In [125]:
dataset.classes

['angry', 'disgusted', 'fearful', 'happy', 'neutral', 'sad', 'surprised']

In [126]:
labels=dataset.targets
len(labels)

28709

In [127]:
from collections import Counter
class_counts=Counter(labels)
class_counts

Counter({0: 3995, 1: 436, 2: 4097, 3: 7215, 4: 4965, 5: 4830, 6: 3171})

In [128]:
#inversing frequency as weights
# class_weights=1.0/class_counts
class_weights={cls: 1/count for cls,count in class_counts.items()}
class_weights

{0: 0.00025031289111389235,
 1: 0.0022935779816513763,
 2: 0.000244081034903588,
 3: 0.0001386001386001386,
 4: 0.0002014098690835851,
 5: 0.00020703933747412008,
 6: 0.000315357931251971}

In [129]:
sample_weights=[class_weights[label]for label in labels]
len(sample_weights)

28709

In [130]:
sampler=WeightedRandomSampler(weights=sample_weights,num_samples=len(sample_weights),replacement=True)

In [131]:
train_loader=DataLoader(dataset=dataset,batch_size=32,sampler=sampler)

In [132]:
from transformers import AutoImageProcessor, AutoModelForImageClassification

processor = AutoImageProcessor.from_pretrained("google/efficientnet-b0")
model = AutoModelForImageClassification.from_pretrained("google/efficientnet-b0")

Loading weights:   0%|          | 0/360 [00:00<?, ?it/s]

In [133]:
model.classifier.in_features

1280

In [134]:
# model.classifier=nn.Sequential(
#     nn.Dropout(p=0.3),
#     nn.Linear(1280,7)
# )

In [135]:
# model.classifier

In [136]:
for param in model.parameters():
    param.requires_grad=False #freezing layers

In [137]:
model.classifier = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(1280,7)
)

In [138]:
# for param in model.classifier.parameters():
#     param.requires_grad=True #unfreezing the classifier

In [139]:
model = model.to(device)

In [140]:
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(params=model.classifier.parameters(),lr=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

In [ ]:
for epoch in range(15):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(pixel_values=images)
        loss = criterion(outputs.logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (outputs.logits.argmax(1) == labels).sum().item()
        total += labels.size(0)

    scheduler.step()
    print(f"Epoch {epoch+1:2d}/15 | "
          f"Loss: {total_loss/len(train_loader):.4f} | "
          f"Acc: {100*correct/total:.1f}%")

In [142]:
print(torch.cuda.is_available())
print(next(model.parameters()).device)

True
cuda:0
